# 🧠 ARC-Challenge: Full Prompting Strategy Benchmark
**Model**: Phi-4-mini-instruct (local) | **Verifier**: Qwen2.5-72B-Instruct (HF API)

## Strategies Compared
| # | Strategy | Description |
|---|----------|-------------|
| 1 | Zero-Shot | Direct answer, no reasoning |
| 2 | Few-Shot | 3 examples before question |
| 3 | Chain-of-Thought | Step-by-step reasoning |
| 4 | Structured CoT | Forced elimination format |
| 5 | Self-Consistency | Majority vote over 5 samples |
| 6 | SLD-VM (no LLM) | Full pipeline, small model VCE only |
| 7 | SLD-VM (with LLM) | Full pipeline, Qwen2.5-72B verifier |

##  CELL 1 — Install & Imports

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install & Imports                                 ║
# ╚══════════════════════════════════════════════════════════════╝
# !pip install -q transformers datasets accelerate huggingface_hub tqdm

import torch, json, re, random, time, hashlib
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from collections import defaultdict, Counter
from enum import Enum
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import InferenceClient, login

CFG = {
    'small_model_id':     'microsoft/Phi-4-mini-instruct',
    'verifier_model':     'Qwen/Qwen2.5-72B-Instruct',
    'llm_conf_threshold': 0.95,   # escalate to LLM if small model conf < this
    'n_samples':          200,     # increase for real results; 10 for quick test
    'seed':               42,
    'max_branches':       3,
    'max_depth':          6,
    'sc_samples':         5,      # self-consistency votes
    'vmb_max_size':       500,
    'vmb_top_k':          3,
    'api_sleep_sec':      2,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
STATS = {'llm_calls': 0, 'llm_errors': 0, 'backtracks': 0, 'vmb_hits': 0, 'nodes_pruned': 0}

print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('✅ Imports ready')

Device : cuda
GPU    : Tesla P100-PCIE-16GB
VRAM   : 17.1 GB
✅ Imports ready


## CELL 2 — HuggingFace Login

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — HuggingFace Login                                ║
# ╚══════════════════════════════════════════════════════════════╝
HF_TOKEN = ""  # ← paste your token
login(HF_TOKEN)
print('✅ HuggingFace login done')

✅ HuggingFace login done


## CELL 3 — Load Small Model

In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load Small Model                                  ║
# ╚══════════════════════════════════════════════════════════════╝
print(f'Loading {CFG["small_model_id"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CFG['small_model_id'])
small_model = AutoModelForCausalLM.from_pretrained(
    CFG['small_model_id'], device_map='auto', torch_dtype=torch.float16)
small_model.eval()
print(f'✅ Model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def _build_chat(system, user):
    msgs = []
    if system: msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': user})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def small_gen(system, user, max_new_tokens=300, temperature=0.0, do_sample=False):
    prompt = _build_chat(system, user)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    L = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = small_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][L:], skip_special_tokens=True).strip()

def small_gen_with_conf(system, user, max_new_tokens=300):
    prompt = _build_chat(system, user)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    L = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = small_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True, output_scores=True)
    scores = out.scores
    conf = (sum(torch.softmax(s[0], dim=-1).max().item() for s in scores) / len(scores)
            if scores else 0.5)
    text = tokenizer.decode(out.sequences[0][L:], skip_special_tokens=True).strip()
    return text, conf

print('✅ Model functions ready')

Loading microsoft/Phi-4-mini-instruct ...


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model loaded | VRAM: 7.7 GB
✅ Model functions ready


## CELL 4 — LLM Verifier (Qwen2.5-72B)

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — LLM Verifier (Qwen2.5-72B)                       ║
# ╚══════════════════════════════════════════════════════════════╝
client = InferenceClient(token=HF_TOKEN)
LLM_AVAILABLE = False
try:
    resp = client.chat_completion(
        model=CFG['verifier_model'],
        messages=[{'role': 'user', 'content': 'Reply with just: OK'}],
        max_tokens=5)
    print(f'✅ LLM Verifier ready | test: {resp.choices[0].message.content.strip()}')
    LLM_AVAILABLE = True
except Exception as e:
    print(f'⚠️  LLM unavailable: {e}')

def llm_call(system, user, max_tokens=150, retries=3):
    if not LLM_AVAILABLE: return None
    for attempt in range(retries):
        try:
            time.sleep(CFG['api_sleep_sec'])
            resp = client.chat_completion(
                model=CFG['verifier_model'],
                messages=[{'role': 'system', 'content': system},
                           {'role': 'user', 'content': user}],
                max_tokens=max_tokens)
            STATS['llm_calls'] += 1
            return resp.choices[0].message.content.strip()
        except Exception as e:
            STATS['llm_errors'] += 1
            if attempt < retries - 1: time.sleep(5 * (attempt + 1))
    return None

print(f'LLM status: {"✅ Active" if LLM_AVAILABLE else "⚠️  Fallback only"}')

⚠️  LLM unavailable: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69a27434-48c7c79c33ae73190b08620d;0b2db5ce-14a8-4a59-b2fd-2001b22d7d51)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
LLM status: ⚠️  Fallback only


## CELL 5 — Load ARC-Challenge Dataset 

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load ARC-Challenge Dataset                        ║
# ╚══════════════════════════════════════════════════════════════╝
print('Loading ARC-Challenge...')
arc_raw = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
arc_list = list(arc_raw)
random.shuffle(arc_list)
arc_samples = arc_list[:CFG['n_samples']]
print(f'✅ ARC-Challenge: {len(arc_samples)} samples loaded')

Loading ARC-Challenge...


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

✅ ARC-Challenge: 200 samples loaded


## CELL 6 — ARC Extractors & Helpers

In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — ARC Extractors & Helpers                         ║
# ╚══════════════════════════════════════════════════════════════╝
def format_arc(sample):
    choices = sample['choices']
    opts = '\n'.join(f'{choices["label"][i]}) {choices["text"][i]}'
                     for i in range(len(choices['text'])))
    return f"{sample['question']}\n\nOptions:\n{opts}"

def arc_gold(sample):
    key = str(sample['answerKey']).strip().upper()
    return {'1': 'A', '2': 'B', '3': 'C', '4': 'D'}.get(key, key)

def extract_arc_pred(text):
    text = text.strip()
    # Priority 1: explicit answer marker
    m = re.search(r'(?:answer(?:\s+is)?|therefore|thus|correct(?:\s+answer)?(?:\s+is)?)[:\s]+([A-D])',
                  text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # Priority 2: bracketed letter
    m = re.search(r'[\(\[]([A-D])[\)\]]', text)
    if m: return m.group(1).upper()
    # Priority 3: last standalone letter
    matches = re.findall(r'\b([A-D])\b', text)
    return matches[-1].upper() if matches else None

def arc_correct(pred, gold):
    if pred is None: return False
    return pred.strip().upper() == gold.strip().upper()

# Verify
assert extract_arc_pred('The correct answer is B.') == 'B'
assert extract_arc_pred('Answer: (C)') == 'C'
assert arc_gold({'answerKey': '2'}) == 'B'
print('✅ Extractors ready')

✅ Extractors ready


## CELL 7 — SLD-VM Pipeline Components

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — SLD-VM Pipeline Components                       ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Graph Structures ──────────────────────────────────────────
class NodeStatus(Enum):
    PENDING = 'pending'; VALID = 'valid'; INVALID = 'invalid'; PRUNED = 'pruned'

@dataclass
class RNode:
    node_id: str; content: str; node_type: str
    parents: List[str] = field(default_factory=list)
    children: List[str] = field(default_factory=list)
    status: NodeStatus = NodeStatus.PENDING
    confidence: float = 0.5; depth: int = 0; branch_id: int = 0; vce_reason: str = ''

@dataclass
class ReasoningGraph:
    problem: str; task: str
    nodes: Dict[str, RNode] = field(default_factory=dict)
    root_id: Optional[str] = None

    def add_node(self, content, node_type, parents=None, depth=0, branch_id=0):
        nid = f'N{len(self.nodes):03d}'
        node = RNode(node_id=nid, content=content, node_type=node_type,
                     parents=parents or [], depth=depth, branch_id=branch_id)
        self.nodes[nid] = node
        for pid in (parents or []):
            if pid in self.nodes: self.nodes[pid].children.append(nid)
        if self.root_id is None: self.root_id = nid
        return nid

    def prune_subtree(self, node_id):
        if node_id not in self.nodes: return
        self.nodes[node_id].status = NodeStatus.PRUNED
        STATS['nodes_pruned'] += 1
        for child in self.nodes[node_id].children: self.prune_subtree(child)

    def valid_nodes(self):
        return [n for n in self.nodes.values() if n.status == NodeStatus.VALID]

    def valid_path_to(self, node_id):
        path, node = [], self.nodes.get(node_id)
        while node:
            path.insert(0, node.content)
            pids = [p for p in node.parents
                    if p in self.nodes and self.nodes[p].status == NodeStatus.VALID]
            node = self.nodes[pids[-1]] if pids else None
        return path

    def summary(self):
        c = Counter(n.status.value for n in self.nodes.values())
        return f'Graph({len(self.nodes)}): ✅{c["valid"]} ❌{c["invalid"]} ✂️{c["pruned"]} ⏳{c["pending"]}'

# ── LTG ───────────────────────────────────────────────────────
LTG_SYS_ARC = """You are a science reasoning decomposer.
Break the question into numbered atomic reasoning steps.
Each step is ONE scientific fact, or eliminates one wrong option with a reason.
Format STRICTLY:
Step 1: [identify what the question asks]
Step 2: [relevant scientific fact]
Step 3: [eliminate wrong options with reasons]
Final: [A, B, C, or D]"""

def ltg_decompose(problem, memory_hint=''):
    user = f'{memory_hint}\n\nProblem:\n{problem}' if memory_hint else f'Problem:\n{problem}'
    text, conf = small_gen_with_conf(LTG_SYS_ARC, user, max_new_tokens=400)
    primitives = []
    for line in text.split('\n'):
        m = re.match(r'^(?:Step\s*\d+|Final)[:\.] *(.+)', line.strip(), re.IGNORECASE)
        if m:
            c = m.group(1).strip()
            if c: primitives.append(c)
    if len(primitives) < 2:
        primitives = [s.strip() for s in re.split(r'[.\n]', text)
                      if len(s.strip()) > 8][:CFG['max_depth']]
    return primitives[:CFG['max_depth']], conf

# ── RGC ───────────────────────────────────────────────────────
def rgc_build(problem, primitives, branch_id=0):
    graph = ReasoningGraph(problem=problem, task='arc')
    if not primitives: return graph
    prev_id = None
    for depth, prim in enumerate(primitives):
        ntype = ('premise' if depth == 0
                 else 'conclusion' if depth == len(primitives) - 1
                 else 'deduction')
        nid = graph.add_node(prim, ntype,
                             parents=[prev_id] if prev_id else [],
                             depth=depth, branch_id=branch_id)
        prev_id = nid
    return graph

def rgc_add_branch(graph, anchor_id, alt_steps, branch_id):
    new_ids, prev_id = [], anchor_id
    anchor_depth = graph.nodes[anchor_id].depth
    for i, step in enumerate(alt_steps):
        depth = anchor_depth + i + 1
        ntype = 'conclusion' if i == len(alt_steps) - 1 else 'deduction'
        nid = graph.add_node(step, ntype, parents=[prev_id],
                             depth=depth, branch_id=branch_id)
        new_ids.append(nid); prev_id = nid
    return new_ids

# ── VCE ───────────────────────────────────────────────────────
def rule_arc_option(content, sample):
    m = re.search(r'(?:answer|choose|option|therefore)[:\s]+([A-D])', content, re.IGNORECASE)
    if m:
        label = m.group(1).upper()
        valid = [l.upper() for l in sample['choices']['label']]
        if label not in valid:
            return False, f'Option {label} not in valid choices {valid}'
    return True, 'ok'

VCE_SM_SYS = """You are a strict logical verifier.
Given a reasoning step and its context, determine if the step is VALID or INVALID.
Reply ONLY with:
VERDICT: VALID
or
VERDICT: INVALID
REASON: one sentence"""

def small_verify(problem, step, prior_nodes):
    prior_text = ' → '.join(n.content for n in prior_nodes[-3:]) or 'none'
    user = (f'Problem: {problem[:300]}\nPrior steps: {prior_text}\nStep to verify: {step}')
    text, conf = small_gen_with_conf(VCE_SM_SYS, user, max_new_tokens=80)
    is_valid = not re.search(r'VERDICT:\s*INVALID', text, re.IGNORECASE)
    return is_valid, conf, text[:120]

VCE_LLM_SYS = """You are an expert verifier for scientific reasoning.
Flag any factual error or logical contradiction.
Reply EXACTLY:
VERDICT: VALID or INVALID
REASON: one sentence"""

def llm_verify(problem, step, prior_nodes):
    prior_text = ' → '.join(n.content for n in prior_nodes[-4:]) or 'none'
    user = f'Problem: {problem[:400]}\nPrior steps: {prior_text}\nStep: {step}'
    resp = llm_call(VCE_LLM_SYS, user, max_tokens=150)
    if resp is None: return True, 'LLM unavailable — pass'
    is_valid = not re.search(r'VERDICT:\s*INVALID', resp, re.IGNORECASE)
    m = re.search(r'REASON:\s*(.+)', resp, re.IGNORECASE)
    reason = m.group(1).strip()[:100] if m else resp[:100]
    return is_valid, f'[LLM] {reason}'

def vce_verify_node(graph, node_id, sample, use_llm=True):
    node = graph.nodes[node_id]
    priors = [graph.nodes[p] for p in node.parents
              if p in graph.nodes and graph.nodes[p].status == NodeStatus.VALID]
    ok, reason = rule_arc_option(node.content, sample)
    if not ok: return False, f'[RULE] {reason}'
    if node.node_type == 'premise':
        node.confidence = 0.9
        return True, '[PREMISE] accepted'
    sm_valid, sm_conf, sm_reason = small_verify(graph.problem, node.content, priors)
    node.confidence = sm_conf
    if use_llm and sm_conf < CFG['llm_conf_threshold'] and LLM_AVAILABLE:
        return llm_verify(graph.problem, node.content, priors)
    return sm_valid, f'[SM|conf={sm_conf:.2f}] {sm_reason[:80]}'

def vce_run_graph(graph, sample, use_llm=True):
    for node in sorted(graph.nodes.values(), key=lambda n: n.depth):
        if node.status != NodeStatus.PENDING: continue
        if any(graph.nodes[p].status in [NodeStatus.PRUNED, NodeStatus.INVALID]
               for p in node.parents if p in graph.nodes):
            continue  # ← remove prune_subtree here
        is_valid, reason = vce_verify_node(graph, node.node_id, sample, use_llm)
        node.vce_reason = reason
        node.status = NodeStatus.VALID if is_valid else NodeStatus.INVALID
        # ← remove prune_subtree here too
    return graph

# ── VMB ───────────────────────────────────────────────────────
@dataclass
class MemoryPattern:
    pid: str; task: str; problem_hash: str
    keywords: List[str]; primitives: List[str]; answer: str
    reliability: float = 1.0; usage_count: int = 0

class VMB:
    def __init__(self, max_size=500):
        self.patterns: Dict[str, MemoryPattern] = {}
        self.max_size = max_size
        self._kw_idx: Dict[str, List[str]] = defaultdict(list)
        self._kw_pool = ['because','causes','result','energy','force','atom','cell',
                         'photosynthesis','gravity','chemical','physical','organism',
                         'process','system','temperature','pressure','evolution',
                         'light','heat','motion','wave','electric','magnetic','nuclear',
                         'species','habitat','ecosystem','adaptation','heredity']

    def _hash(self, text): return hashlib.md5(text.strip().lower().encode()).hexdigest()[:10]
    def _keywords(self, text): return [kw for kw in self._kw_pool if kw in text.lower()]

    def store(self, problem, primitives, answer):
        if len(self.patterns) >= self.max_size:
            worst = min(self.patterns.values(), key=lambda p: p.reliability)
            for kw in worst.keywords: self._kw_idx[kw] = [x for x in self._kw_idx[kw] if x != worst.pid]
            del self.patterns[worst.pid]
        pid = f'P{len(self.patterns):04d}'
        kws = self._keywords(problem)
        pat = MemoryPattern(pid=pid, task='arc', problem_hash=self._hash(problem),
                            keywords=kws, primitives=primitives, answer=answer)
        self.patterns[pid] = pat
        for kw in kws: self._kw_idx[kw].append(pid)

    def retrieve(self, problem, top_k=3):
        kws = self._keywords(problem)
        scores = defaultdict(float)
        for kw in kws:
            for pid in self._kw_idx.get(kw, []):
                if pid in self.patterns:
                    p = self.patterns[pid]
                    scores[pid] += p.reliability * (1 + 0.05 * p.usage_count)
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        result = []
        for pid, _ in top:
            if pid in self.patterns:
                self.patterns[pid].usage_count += 1
                result.append(self.patterns[pid])
        if result: STATS['vmb_hits'] += 1
        return result

    def update_reliability(self, problem, success):
        phash = self._hash(problem)
        for p in self.patterns.values():
            if p.problem_hash == phash:
                p.reliability = min(1.0, p.reliability * 1.15) if success else max(0.05, p.reliability * 0.75)

    def stats(self):
        if not self.patterns: return 'VMB: empty'
        avg_r = sum(p.reliability for p in self.patterns.values()) / len(self.patterns)
        return f'VMB: {len(self.patterns)} patterns | avg_rel={avg_r:.2f} | hits={STATS["vmb_hits"]}'

# ── Deliberation ──────────────────────────────────────────────
REGEN_SYS = """A reasoning step about this science question was rejected.
Suggest 2 ALTERNATIVE reasoning steps from the valid steps so far.
Focus on scientific facts and eliminating wrong answer choices.
Format:
Alt 1: [step]
Alt 2: [step]"""

FINAL_SYS = """Given verified reasoning steps about this science question,
state which option (A, B, C, or D) is correct.
Output ONLY the single letter. Nothing else."""

def deliberate_alternatives(problem, valid_steps, failed_step):
    valid_text = ' → '.join(valid_steps[-3:]) if valid_steps else 'none yet'
    user = (f'Problem: {problem[:400]}\nValid steps so far: {valid_text}\n'
            f'Failed step: {failed_step}\nSuggest alternatives:')
    text = small_gen(REGEN_SYS, user, max_new_tokens=200)
    alts = []
    for line in text.split('\n'):
        m = re.match(r'Alt\s*\d+:\s*(.+)', line.strip(), re.IGNORECASE)
        if m: alts.append(m.group(1).strip())
    return alts[:CFG['max_branches']]

def deliberate_extract_answer(graph, sample):
    valid = graph.valid_nodes()
    if not valid: return None
    for node in sorted(valid, key=lambda n: n.depth, reverse=True):
        if node.node_type == 'conclusion':
            ans = extract_arc_pred(node.content)
            if ans: return ans
    steps = ' → '.join(n.content for n in sorted(valid, key=lambda n: n.depth))
    choices = sample['choices']
    opts = '\n'.join(f'{choices["label"][i]}) {choices["text"][i]}' for i in range(len(choices['text'])))
    out = small_gen(FINAL_SYS,
                    f'Question: {graph.problem[:300]}\nOptions:\n{opts}\nVerified steps: {steps}',
                    max_new_tokens=20)
    return extract_arc_pred(out)

print('✅ All pipeline components ready')

✅ All pipeline components ready


## CELL 8 — SLD-VM Runner (with use_llm flag)

In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — SLD-VM Runner (with use_llm flag)                 ║
# ╚══════════════════════════════════════════════════════════════╝
vmb_with_llm    = VMB(max_size=CFG['vmb_max_size'])
vmb_without_llm = VMB(max_size=CFG['vmb_max_size'])

@dataclass
class PipelineResult:
    answer: Optional[str]; graph: ReasoningGraph
    used_vmb: bool = False; backtracks: int = 0
    llm_calls: int = 0; conf: float = 0.0

def sldvm_arc(problem, sample, use_llm=True, verbose=False):
    pre_llm = STATS['llm_calls']
    backtracks = 0
    vmb = vmb_with_llm if use_llm else vmb_without_llm

    # Stage 1: VMB Retrieval
    patterns = vmb.retrieve(problem, top_k=CFG['vmb_top_k'])
    used_vmb = len(patterns) > 0
    memory_hint = ''
    if patterns:
        hint = ' → '.join(patterns[0].primitives[:3])
        memory_hint = f'[Similar problem solved via: {hint}]'

    # Stage 2: LTG
    primitives, ltg_conf = ltg_decompose(problem, memory_hint)
    if verbose: print(f'  LTG: {len(primitives)} steps | conf={ltg_conf:.2f}')
    if not primitives:
        return PipelineResult(answer=None, graph=ReasoningGraph(problem=problem, task='arc'), used_vmb=used_vmb)

    # Stage 3: RGC
    graph = rgc_build(problem, primitives)

    # Stage 4: VCE
    graph = vce_run_graph(graph, sample, use_llm=use_llm)
    if verbose: print(f'  VCE: {graph.summary()}')

    # Stage 5: Deliberation
    invalid_nodes = [n for n in graph.nodes.values() if n.status == NodeStatus.INVALID]
    for inv_node in invalid_nodes[:CFG['max_branches']]:
        if backtracks >= CFG['max_branches']: break
        valid_ancestors = [graph.nodes[p] for p in inv_node.parents
                           if p in graph.nodes and graph.nodes[p].status == NodeStatus.VALID]
        if not valid_ancestors: continue
        anchor = valid_ancestors[-1]
        alts = deliberate_alternatives(problem, graph.valid_path_to(anchor.node_id), inv_node.content)
        if alts:
            backtracks += 1; STATS['backtracks'] += 1
            new_ids = rgc_add_branch(graph, anchor.node_id, alts, branch_id=backtracks)
            for nid in new_ids:
                if graph.nodes[nid].status == NodeStatus.PENDING:
                    ok, reason = vce_verify_node(graph, nid, sample, use_llm)
                    graph.nodes[nid].vce_reason = reason
                    graph.nodes[nid].status = NodeStatus.VALID if ok else NodeStatus.INVALID

    # ← ADD HERE: prune remaining INVALID after deliberation is done
    for node in list(graph.nodes.values()):
        if node.status == NodeStatus.INVALID:
            graph.prune_subtree(node.node_id)

    # Stage 6: Extract answer
    answer = deliberate_extract_answer(graph, sample)

    # Stage 7: VMB Store
    if answer is not None:
        valid_prims = [n.content for n in sorted(graph.valid_nodes(), key=lambda n: n.depth)]
        if valid_prims: vmb.store(problem, valid_prims, answer)

    valid = graph.valid_nodes()
    conf = sum(n.confidence for n in valid) / len(valid) if valid else 0.0
    return PipelineResult(answer=answer, graph=graph, used_vmb=used_vmb,
                          backtracks=backtracks,
                          llm_calls=STATS['llm_calls'] - pre_llm, conf=conf)

print('✅ SLD-VM runner ready (supports use_llm=True/False)')

✅ SLD-VM runner ready (supports use_llm=True/False)


## CELL 9 — Strategy 1: Zero-Shot 

In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Strategy 1: Zero-Shot                             ║
# ╚══════════════════════════════════════════════════════════════╝
ZS_SYS = 'You are a science expert. Output ONLY the letter (A, B, C, or D) of the correct answer.'

correct, null_count = 0, 0
zs_records = []
for i, s in enumerate(tqdm(arc_samples, desc='Zero-Shot')):
    out  = small_gen(ZS_SYS, format_arc(s), max_new_tokens=10)
    pred = extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    zs_records.append({'pred': pred, 'gold': gold, 'ok': ok, 'out': out, 's': s})

zs_acc = correct / len(arc_samples)
print(f'\n✅ Zero-Shot: {zs_acc:.1%} | Null={null_count}')

Zero-Shot: 100%|██████████| 200/200 [01:30<00:00,  2.22it/s]


✅ Zero-Shot: 82.5% | Null=1


## CELL 10 — Strategy 2: Few-Shot (3 examples)

In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Strategy 2: Few-Shot (3 examples)                ║
# ╚══════════════════════════════════════════════════════════════╝
FEW_SHOT_SYS = 'You are a science expert. Study the examples then answer. Output ONLY the letter (A, B, C, or D).'

FEW_SHOT_EXAMPLES = """Example 1:
Which process do plants use to make food?
Options:
A) Respiration
B) Photosynthesis
C) Digestion
D) Fermentation
Answer: B

Example 2:
What happens to the volume of a gas when it is heated?
Options:
A) It decreases
B) It stays the same
C) It increases
D) It becomes liquid
Answer: C

Example 3:
Which of the following is a producer in a food chain?
Options:
A) Lion
B) Grass
C) Deer
D) Eagle
Answer: B

Now answer this:"""

correct, null_count = 0, 0
fs_records = []
for i, s in enumerate(tqdm(arc_samples, desc='Few-Shot')):
    user = f'{FEW_SHOT_EXAMPLES}\n{format_arc(s)}'
    out  = small_gen(FEW_SHOT_SYS, user, max_new_tokens=10)
    pred = extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    fs_records.append({'pred': pred, 'gold': gold, 'ok': ok, 'out': out, 's': s})

fs_acc = correct / len(arc_samples)
print(f'\n✅ Few-Shot: {fs_acc:.1%} | Null={null_count}')

Few-Shot: 100%|██████████| 200/200 [01:48<00:00,  1.85it/s]


✅ Few-Shot: 79.0% | Null=2


## CELL 11 — Strategy 3: Chain-of-Thought

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Strategy 3: Chain-of-Thought                     ║
# ╚══════════════════════════════════════════════════════════════╝
COT_SYS = """You are a science expert.
Reason step by step. Eliminate wrong answers. End with: Answer: [A/B/C/D]"""

COT_EX = """Example:
Which process do plants use to make food?
A) Respiration  B) Photosynthesis  C) Digestion  D) Fermentation
Step 1: Plants convert sunlight + CO2 + water into glucose.
Step 2: That process is photosynthesis. Respiration/digestion/fermentation don't produce food.
Answer: B
"""

correct, null_count = 0, 0
cot_records = []
for i, s in enumerate(tqdm(arc_samples, desc='Chain-of-Thought')):
    out  = small_gen(COT_SYS, f'{COT_EX}\n{format_arc(s)}', max_new_tokens=300)
    pred = extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    cot_records.append({'pred': pred, 'gold': gold, 'ok': ok, 'out': out, 's': s})

cot_acc = correct / len(arc_samples)
print(f'\n✅ Chain-of-Thought: {cot_acc:.1%} | Null={null_count}')

Chain-of-Thought: 100%|██████████| 200/200 [17:27<00:00,  5.24s/it]


✅ Chain-of-Thought: 81.5% | Null=1


##  CELL 12 — Strategy 4: Structured CoT (forced elimination)

In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Strategy 4: Structured CoT (forced elimination)  ║
# ╚══════════════════════════════════════════════════════════════╝
SCOT_SYS = """You are a science expert. Use this EXACT format:
QUESTION ASKS: [one sentence — what is being asked]
KEY FACT: [one relevant scientific fact]
ELIMINATE: [list wrong options with one-word reason each]
ANSWER: [single letter A/B/C/D]"""

correct, null_count = 0, 0
scot_records = []
for i, s in enumerate(tqdm(arc_samples, desc='Structured CoT')):
    out  = small_gen(SCOT_SYS, format_arc(s), max_new_tokens=200)
    # Try to extract from ANSWER: field first
    m = re.search(r'ANSWER:\s*([A-D])', out, re.IGNORECASE)
    pred = m.group(1).upper() if m else extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    scot_records.append({'pred': pred, 'gold': gold, 'ok': ok, 'out': out, 's': s})

scot_acc = correct / len(arc_samples)
print(f'\n✅ Structured CoT: {scot_acc:.1%} | Null={null_count}')

Structured CoT: 100%|██████████| 200/200 [16:39<00:00,  5.00s/it]


✅ Structured CoT: 83.0% | Null=0


##  CELL 13 — Strategy 5: Self-Consistency (majority vote)

In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Strategy 5: Self-Consistency (majority vote)     ║
# ╚══════════════════════════════════════════════════════════════╝
SC_SYS = """You are a science expert.
Reason step by step then give your answer as a single letter (A/B/C/D)."""

correct, null_count = 0, 0
sc_records = []
for i, s in enumerate(tqdm(arc_samples, desc='Self-Consistency')):
    votes = []
    for _ in range(CFG['sc_samples']):
        out  = small_gen(SC_SYS, format_arc(s), max_new_tokens=200, temperature=0.7, do_sample=True)
        pred = extract_arc_pred(out)
        if pred: votes.append(pred)
    # Majority vote
    final = Counter(votes).most_common(1)[0][0] if votes else None
    gold  = arc_gold(s)
    ok    = arc_correct(final, gold)
    if ok: correct += 1
    if final is None: null_count += 1
    sc_records.append({'pred': final, 'gold': gold, 'ok': ok, 'votes': votes, 's': s})

sc_acc = correct / len(arc_samples)
print(f'\n✅ Self-Consistency ({CFG["sc_samples"]} votes): {sc_acc:.1%} | Null={null_count}')

Self-Consistency: 100%|██████████| 200/200 [44:56<00:00, 13.48s/it]


✅ Self-Consistency (5 votes): 79.0% | Null=0


## CELL 14 — Strategy 6: SLD-VM WITHOUT LLM 

In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Strategy 6: SLD-VM WITHOUT LLM                   ║
# ╚══════════════════════════════════════════════════════════════╝
for k in STATS: STATS[k] = 0

correct, null_count = 0, 0
sldvm_nollm_records = []
print('Running SLD-VM (NO LLM)...\n')

for i, s in enumerate(tqdm(arc_samples, desc='SLD-VM No-LLM')):
    problem = format_arc(s)
    result  = sldvm_arc(problem, s, use_llm=False)
    pred    = result.answer
    gold    = arc_gold(s)
    ok      = arc_correct(pred, gold)
    if ok: correct += 1; vmb_without_llm.update_reliability(problem, True)
    else: vmb_without_llm.update_reliability(problem, False)
    if pred is None: null_count += 1
    sldvm_nollm_records.append({'pred': pred, 'gold': gold, 'ok': ok, 's': s, 'result': result})
    if i < 3: print(f'  [{i+1}] pred={pred} gold={gold} ok={ok} bt={result.backtracks} | {result.graph.summary()}')

sldvm_nollm_acc = correct / len(arc_samples)
print(f'\n✅ SLD-VM (No LLM): {sldvm_nollm_acc:.1%} | Null={null_count}')
print(f'   Backtracks: {STATS["backtracks"]} | Nodes pruned: {STATS["nodes_pruned"]}')
print(f'   {vmb_without_llm.stats()}')

Running SLD-VM (NO LLM)...



SLD-VM No-LLM:   0%|          | 1/200 [00:13<44:17, 13.36s/it]

  [1] pred=A gold=A ok=True bt=0 | Graph(4): ✅4 ❌0 ✂️0 ⏳0


SLD-VM No-LLM:   1%|          | 2/200 [00:31<54:15, 16.44s/it]

  [2] pred=B gold=B ok=True bt=1 | Graph(6): ✅2 ❌0 ✂️4 ⏳0


SLD-VM No-LLM:   2%|▏         | 3/200 [00:57<1:07:53, 20.68s/it]

  [3] pred=B gold=C ok=False bt=1 | Graph(6): ✅3 ❌0 ✂️3 ⏳0


SLD-VM No-LLM: 100%|██████████| 200/200 [58:38<00:00, 17.59s/it] 


✅ SLD-VM (No LLM): 82.0% | Null=1
   Backtracks: 111 | Nodes pruned: 336
   VMB: 199 patterns | avg_rel=0.96 | hits=107


## CELL 15 — Strategy 7: SLD-VM WITH LLM

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Strategy 7: SLD-VM WITH LLM                      ║
# ╚══════════════════════════════════════════════════════════════╝
for k in STATS: STATS[k] = 0

correct, null_count = 0, 0
sldvm_llm_records = []
print('Running SLD-VM (WITH LLM)...\n')

for i, s in enumerate(tqdm(arc_samples, desc='SLD-VM With-LLM')):
    problem = format_arc(s)
    result  = sldvm_arc(problem, s, use_llm=True)
    pred    = result.answer
    gold    = arc_gold(s)
    ok      = arc_correct(pred, gold)
    if ok: correct += 1; vmb_with_llm.update_reliability(problem, True)
    else: vmb_with_llm.update_reliability(problem, False)
    if pred is None: null_count += 1
    sldvm_llm_records.append({'pred': pred, 'gold': gold, 'ok': ok, 's': s, 'result': result})
    if i < 3: print(f'  [{i+1}] pred={pred} gold={gold} ok={ok} bt={result.backtracks} llm={result.llm_calls} | {result.graph.summary()}')

sldvm_llm_acc = correct / len(arc_samples)
print(f'\n✅ SLD-VM (With LLM): {sldvm_llm_acc:.1%} | Null={null_count}')
print(f'   Backtracks: {STATS["backtracks"]} | LLM calls: {STATS["llm_calls"]} | Nodes pruned: {STATS["nodes_pruned"]}')
print(f'   {vmb_with_llm.stats()}')

Running SLD-VM (WITH LLM)...



SLD-VM With-LLM:   0%|          | 1/200 [00:13<43:13, 13.03s/it]

  [1] pred=A gold=A ok=True bt=0 llm=0 | Graph(4): ✅4 ❌0 ✂️0 ⏳0


SLD-VM With-LLM:   1%|          | 2/200 [00:31<53:33, 16.23s/it]

  [2] pred=B gold=B ok=True bt=1 llm=0 | Graph(6): ✅2 ❌0 ✂️4 ⏳0


SLD-VM With-LLM:   2%|▏         | 3/200 [00:57<1:07:37, 20.60s/it]

  [3] pred=B gold=C ok=False bt=1 llm=0 | Graph(6): ✅3 ❌0 ✂️3 ⏳0


SLD-VM With-LLM: 100%|██████████| 200/200 [58:31<00:00, 17.56s/it] 


✅ SLD-VM (With LLM): 82.0% | Null=1
   Backtracks: 111 | LLM calls: 0 | Nodes pruned: 336
   VMB: 199 patterns | avg_rel=0.96 | hits=107


##  CELL 16 — Final Results Table

In [16]:
# ╔══════════════════════════════════════════════════════════════╗  
# ║  CELL 16 — Final Results Table                              ║
# ╚══════════════════════════════════════════════════════════════╝
results = [
    ('1. Zero-Shot',               zs_acc,           zs_records),
    ('2. Few-Shot (3-ex)',         fs_acc,           fs_records),
    ('3. Chain-of-Thought',        cot_acc,          cot_records),
    ('4. Structured CoT',          scot_acc,         scot_records),
    ('5. Self-Consistency (5x)',   sc_acc,           sc_records),
    ('6. SLD-VM  (No LLM)',        sldvm_nollm_acc,  sldvm_nollm_records),
    ('7. SLD-VM  (With LLM)',      sldvm_llm_acc,    sldvm_llm_records),
]

best_baseline = max(zs_acc, fs_acc, cot_acc, scot_acc, sc_acc)

print('=' * 65)
print(f'  ARC-Challenge Benchmark — {len(arc_samples)} samples')
print(f'  Model: {CFG["small_model_id"]}')
print(f'  Verifier: {CFG["verifier_model"]}')
print('=' * 65)
print(f'{"Strategy":<30} {"Acc":>7} {"Δ vs Best BL":>13}')
print('-' * 65)
for name, acc, _ in results:
    delta = acc - best_baseline
    flag  = '🏆' if acc == max(r[1] for r in results) else ''
    print(f'{name:<30} {acc:>6.1%}  {delta:>+8.1%}  {flag}')
print('=' * 65)
print(f'Random baseline (4-choice):     25.0%')
print(f'Best prompting baseline:       {best_baseline:.1%}')
print('=' * 65)

# LLM impact
llm_delta = sldvm_llm_acc - sldvm_nollm_acc
print(f'\n── LLM Verifier Impact ──────────────────────────────────')
print(f'  SLD-VM No LLM  : {sldvm_nollm_acc:.1%}')
print(f'  SLD-VM With LLM: {sldvm_llm_acc:.1%}')
print(f'  LLM adds       : {llm_delta:+.1%}')
print(f'  LLM calls used : {STATS["llm_calls"]}')

  ARC-Challenge Benchmark — 200 samples
  Model: microsoft/Phi-4-mini-instruct
  Verifier: Qwen/Qwen2.5-72B-Instruct
Strategy                           Acc  Δ vs Best BL
-----------------------------------------------------------------
1. Zero-Shot                    82.5%     -0.5%  
2. Few-Shot (3-ex)              79.0%     -4.0%  
3. Chain-of-Thought             81.5%     -1.5%  
4. Structured CoT               83.0%     +0.0%  🏆
5. Self-Consistency (5x)        79.0%     -4.0%  
6. SLD-VM  (No LLM)             82.0%     -1.0%  
7. SLD-VM  (With LLM)           82.0%     -1.0%  
Random baseline (4-choice):     25.0%
Best prompting baseline:       83.0%

── LLM Verifier Impact ──────────────────────────────────
  SLD-VM No LLM  : 82.0%
  SLD-VM With LLM: 82.0%
  LLM adds       : +0.0%
  LLM calls used : 0


## CELL 17 — Win/Loss Analysis Between All Strategies

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 17 — Win/Loss Analysis Between All Strategies         ║
# ╚══════════════════════════════════════════════════════════════╝
print('── SLD-VM (with LLM) vs Each Strategy ──────────────────')
sldvm_ok = {i for i, r in enumerate(sldvm_llm_records) if r['ok']}

for name, _, records in results[:-1]:  # compare against all except SLD-VM with LLM itself
    other_ok = {i for i, r in enumerate(records) if r['ok']}
    wins   = sldvm_ok - other_ok
    losses = other_ok - sldvm_ok
    both   = sldvm_ok & other_ok
    print(f'\n  vs {name}')
    print(f'    SLD-VM wins   : {len(wins)}')
    print(f'    SLD-VM loses  : {len(losses)}')
    print(f'    Both correct  : {len(both)}')

print('\n── Where SLD-VM (with LLM) wins vs CoT ─────────────────')
cot_ok = {i for i, r in enumerate(cot_records) if r['ok']}
wins = sldvm_ok - cot_ok
for idx in list(wins)[:5]:
    r = sldvm_llm_records[idx]
    print(f'  Q: {r["s"]["question"][:80]}')
    print(f'     CoT={cot_records[idx]["pred"]} SLD-VM={r["pred"]} Gold={r["gold"]}')

── SLD-VM (with LLM) vs Each Strategy ──────────────────

  vs 1. Zero-Shot
    SLD-VM wins   : 14
    SLD-VM loses  : 15
    Both correct  : 150

  vs 2. Few-Shot (3-ex)
    SLD-VM wins   : 19
    SLD-VM loses  : 13
    Both correct  : 145

  vs 3. Chain-of-Thought
    SLD-VM wins   : 21
    SLD-VM loses  : 20
    Both correct  : 143

  vs 4. Structured CoT
    SLD-VM wins   : 16
    SLD-VM loses  : 18
    Both correct  : 148

  vs 5. Self-Consistency (5x)
    SLD-VM wins   : 19
    SLD-VM loses  : 13
    Both correct  : 145

  vs 6. SLD-VM  (No LLM)
    SLD-VM wins   : 0
    SLD-VM loses  : 0
    Both correct  : 164

── Where SLD-VM (with LLM) wins vs CoT ─────────────────
  Q: Chemical weathering occurs when minerals in rocks are changed chemically. Which 
     CoT=D SLD-VM=B Gold=B
  Q: Students shake salt into a jar of water, and they shake pepper into a second jar
     CoT=C SLD-VM=D Gold=D
  Q: What role do ribosomes play in the central dogma of molecular biology?
     CoT=B SLD

## CELL 18 — Failure Analysis: SLD-VM With LLM 

In [18]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Failure Analysis: SLD-VM With LLM                ║
# ╚══════════════════════════════════════════════════════════════╝
failures = [r for r in sldvm_llm_records if not r['ok']]
print(f'═'*60)
print(f'SLD-VM (With LLM) — {len(failures)} failures')
print(f'═'*60)
for i, r in enumerate(failures[:5]):
    res = r['result']
    print(f'\n[{i+1}] pred={r["pred"]}  gold={r["gold"]}')
    print(f'  Q: {r["s"]["question"][:100]}')
    print(f'  Choices: {r["s"]["choices"]["text"]}')
    print(f'  {res.graph.summary()}')
    for nid, node in list(res.graph.nodes.items())[:4]:
        icon = {'valid': '✅', 'invalid': '❌', 'pruned': '✂️', 'pending': '⏳'}[node.status.value]
        print(f'    {icon} [{node.node_type:10s}] {node.content[:70]}')
        if node.vce_reason: print(f'       VCE: {node.vce_reason[:80]}')

════════════════════════════════════════════════════════════
SLD-VM (With LLM) — 36 failures
════════════════════════════════════════════════════════════

[1] pred=B  gold=C
  Q: Which object in the solar system is orbited by a belt of asteroids?
  Choices: ['Pluto', 'Saturn', 'the Sun', 'the Moon']
  Graph(6): ✅3 ❌0 ✂️3 ⏳0
    ✅ [premise   ] Identify what the question asks
       VCE: [PREMISE] accepted
    ✅ [deduction ] Relevant scientific fact
       VCE: [SM|conf=0.73] VERDICT: VALID
REASON: Saturn is known to have a prominent ring s
    ✅ [deduction ] Eliminate wrong options with reasons
       VCE: [SM|conf=0.73] VERDICT: VALID
REASON: The Sun is not an object that is orbited b
    ✂️ [conclusion] None of the options provided (A, B, C, or D) are correct. The correct 
       VCE: [SM|conf=0.70] VERDICT: INVALID
REASON: The step to verify is not a reasoning st

[2] pred=B  gold=A
  Q: Nick is making a model to show how human skin covers the body. What material would work best?
  C

## CELL 19 — Save Results to JSON

In [19]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Save Results to JSON                             ║
# ╚══════════════════════════════════════════════════════════════╝
output = {
    'model': CFG['small_model_id'],
    'verifier': CFG['verifier_model'],
    'n_samples': len(arc_samples),
    'results': {
        'zero_shot':        zs_acc,
        'few_shot':         fs_acc,
        'chain_of_thought': cot_acc,
        'structured_cot':   scot_acc,
        'self_consistency': sc_acc,
        'sldvm_no_llm':     sldvm_nollm_acc,
        'sldvm_with_llm':   sldvm_llm_acc,
    },
    'llm_impact': sldvm_llm_acc - sldvm_nollm_acc,
    'best_baseline': best_baseline,
    'sldvm_vs_best_baseline': sldvm_llm_acc - best_baseline,
    'pipeline_stats': dict(STATS),
}
with open('arc_benchmark_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('✅ Results saved to arc_benchmark_results.json')
print(json.dumps(output['results'], indent=2))

✅ Results saved to arc_benchmark_results.json
{
  "zero_shot": 0.825,
  "few_shot": 0.79,
  "chain_of_thought": 0.815,
  "structured_cot": 0.83,
  "self_consistency": 0.79,
  "sldvm_no_llm": 0.82,
  "sldvm_with_llm": 0.82
}
